# D390 — Snowflake data types: complete practical

Data type choice controls correctness, comparison behavior, storage, conversion, and which SQL functions can be used. This notebook teaches every current Snowflake data-type category through runnable examples.

The core labs are self-contained. JSON and XML are embedded with Snowflake dollar-quoted strings (`$$...$$`), which serve the same purpose as triple-quoted multiline strings and paste cleanly into a Snowsight SQL worksheet. An optional section connects to the files and stage from `D381_StagePractical.ipynb`.

## 1. Complete type map

| Category | Canonical types | Important aliases or notes |
|---|---|---|
| Fixed numeric | `NUMBER(p,s)` | `DECIMAL`, `DEC`, and `NUMERIC` are aliases; integer names map to `NUMBER(38,0)` |
| Decimal floating point | `DECFLOAT` | Exact decimal with dynamic base-10 exponent and up to 38 significant digits |
| Binary floating point | `FLOAT` | `DOUBLE`, `DOUBLE PRECISION`, and `REAL` are aliases |
| Text | `VARCHAR(n)` | `STRING`, `TEXT`, `VARCHAR2`, `NVARCHAR`, and most character names are aliases |
| Binary | `BINARY(n)` | `VARBINARY` is an alias |
| Logical | `BOOLEAN` | Values are `TRUE`, `FALSE`, or SQL `NULL`/unknown |
| Date and time | `DATE`, `TIME`, `TIMESTAMP_NTZ`, `TIMESTAMP_LTZ`, `TIMESTAMP_TZ` | `DATETIME` is an NTZ alias; `TIMESTAMP` mapping is session-controlled |
| Semi-structured | `VARIANT`, semi-structured `OBJECT`, semi-structured `ARRAY` | Values can have different nested shapes and types by row |
| Structured | `OBJECT(field type, ...)`, `ARRAY(type)`, `MAP(key_type, value_type)` | Field/element/value types are declared |
| Geospatial | `GEOGRAPHY`, `GEOMETRY` | Spherical Earth coordinates versus planar coordinates |
| Unstructured reference | `FILE` | Stores staged-file metadata/reference, not the file bytes |
| Identifier | `UUID` | Native 128-bit UUID; uniqueness is not automatically enforced |
| Vector | `VECTOR(INT,n)`, `VECTOR(FLOAT,n)` | Fixed dimension, maximum 4096 |
| User-defined | A schema object based on one Snowflake type | Reusable scalar or structured domain name |
| Iceberg-only placeholder | `UNKNOWN` | Iceberg v3 only; has no storage and reads as SQL `NULL` |

The word `OBJECT` or `ARRAY` alone means the flexible semi-structured form. Adding field or element declarations creates the structured form. File formats such as CSV, JSON, XML, Avro, ORC, and Parquet describe file parsing; they are not table-column data types.

Source: [Snowflake data-type summary](https://docs.snowflake.com/en/sql-reference/intro-summary-data-types).

## 2. Lab environment

Use an authorized role. `SYSADMIN` is used for a simple training account; adapt it to your organization. Every table name is confined to `D39_TABLE_LAB.DATATYPES`.

```sql
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS D39_TABLE_LAB;
CREATE SCHEMA IF NOT EXISTS D39_TABLE_LAB.DATATYPES;
CREATE WAREHOUSE IF NOT EXISTS D39_LAB_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE D39_LAB_WH;
USE DATABASE D39_TABLE_LAB;
USE SCHEMA DATATYPES;

ALTER SESSION SET TIMEZONE = 'Asia/Kolkata';
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(),
       CURRENT_TIMESTAMP() AS SESSION_TIMESTAMP;
SHOW PARAMETERS LIKE 'TIMEZONE' IN SESSION;
```

## 3. Inspect and convert types

Use `SYSTEM$TYPEOF` to inspect the resolved SQL type. `CAST(value AS type)` and `value::type` raise an error for invalid input. `TRY_TO_*` functions return SQL `NULL`, which is useful when profiling imperfect source data. Explicit conversion makes production SQL easier to review than relying on coercion.

```sql
SELECT
  SYSTEM$TYPEOF(42)                         AS INTEGER_LITERAL_TYPE,
  SYSTEM$TYPEOF(42.25)                      AS DECIMAL_LITERAL_TYPE,
  SYSTEM$TYPEOF('42')                       AS STRING_LITERAL_TYPE,
  SYSTEM$TYPEOF('2026-09-07'::DATE)         AS DATE_TYPE,
  SYSTEM$TYPEOF(PARSE_JSON('{"x":1}')) AS JSON_TYPE;

SELECT
  CAST('123.45' AS NUMBER(10,2)) AS CAST_FORM,
  '123.45'::NUMBER(10,2)         AS COLON_FORM,
  TRY_TO_NUMBER('not-a-number')  AS SAFE_FAILURE,
  TRY_TO_DATE('31/02/2026', 'DD/MM/YYYY') AS INVALID_DATE;

-- These are deliberately commented because they fail.
-- SELECT 'not-a-number'::NUMBER;
-- SELECT '2026-99-99'::DATE;
```

Source: [data type conversion](https://docs.snowflake.com/en/sql-reference/data-type-conversion).

## 4. Numeric types

Use `NUMBER(p,s)` for money, identifiers, counts, and measurements that require a fixed decimal rule. Precision is total digits; scale is digits to the right of the decimal point. Use `FLOAT` for approximate scientific or statistical calculations. Use `DECFLOAT` when values need exact decimal semantics but their scale or exponent varies widely.

```sql
CREATE OR REPLACE TABLE NUMERIC_TYPES_LAB (
  ID INTEGER,
  QUANTITY NUMBER(10,0),
  UNIT_PRICE NUMBER(12,2),
  PRECISE_RATIO NUMBER(20,10),
  APPROX_READING FLOAT,
  WIDE_EXACT_VALUE DECFLOAT
);

INSERT INTO NUMERIC_TYPES_LAB VALUES
  (1, 3, 199.995, 0.3333333333, 0.1, '1.234567890123456789e-25'::DECFLOAT),
  (2, 2,  49.994, 0.6666666667, 'NaN'::FLOAT, DECFLOAT '9.99e500');

SELECT *, QUANTITY * UNIT_PRICE AS LINE_TOTAL
FROM NUMERIC_TYPES_LAB
WHERE UNIT_PRICE >= 50
ORDER BY ID;

SELECT
  12.345::NUMBER(5,2) AS ROUNDED_TO_SCALE,
  SYSTEM$TYPEOF(1::INTEGER) AS INTEGER_ALIAS_RESOLVES_TO,
  0.1::FLOAT + 0.2::FLOAT AS BINARY_FLOAT_SUM,
  ABS((0.1::FLOAT + 0.2::FLOAT) - 0.3::FLOAT) < 1e-12
    AS APPROXIMATELY_EQUAL,
  '1.0000000000000000000000000000000000001'::DECFLOAT
    AS EXACT_DECIMAL_FLOAT;

UPDATE NUMERIC_TYPES_LAB
SET UNIT_PRICE = UNIT_PRICE * 0.90
WHERE ID = 1;

DELETE FROM NUMERIC_TYPES_LAB WHERE ID = 2;
SELECT * FROM NUMERIC_TYPES_LAB;

-- Fails because NUMBER(5,2) allows only three digits left of the decimal.
-- SELECT 12345.67::NUMBER(5,2);
```

`NUMBER`, `DECIMAL`, and `NUMERIC` are synonymous. `INT`, `INTEGER`, `BIGINT`, `SMALLINT`, `TINYINT`, and `BYTEINT` all resolve to `NUMBER(38,0)` rather than different integer ranges. Source: [numeric types](https://docs.snowflake.com/en/sql-reference/data-types-numeric).

## 5. Text and binary types

Snowflake strings use UTF-8 Unicode. `CHAR` is not padded like it is in some databases. A declared maximum can protect downstream tools, but storage is based on actual content. `BINARY` stores bytes; its displayed hexadecimal text is a representation of those bytes.

```sql
CREATE OR REPLACE TABLE TEXT_BINARY_LAB (
  ID INTEGER,
  SHORT_CODE CHAR(5),
  CUSTOMER_NAME VARCHAR(100),
  NOTES TEXT,
  UTF8_TEXT STRING,
  RAW_BYTES BINARY(32),
  OTHER_BYTES VARBINARY
);

INSERT INTO TEXT_BINARY_LAB VALUES
  (1, 'IN', 'Asha Rao', $$Customer said: 'deliver carefully'$$,
   'தமிழ் • हिन्दी • 日本語',
   TO_BINARY('48656C6C6F', 'HEX'),
   MD5_BINARY('Asha Rao')),
  (2, 'US', 'Ben Lee', 'Second row', 'Snowflake ❄',
   TO_BINARY('U25vd2ZsYWtl', 'BASE64'),
   MD5_BINARY('Ben Lee'));

SELECT ID, SHORT_CODE, CUSTOMER_NAME, UTF8_TEXT,
       LENGTH(UTF8_TEXT) AS CHARACTER_COUNT,
       OCTET_LENGTH(UTF8_TEXT) AS UTF8_BYTE_COUNT,
       TO_VARCHAR(RAW_BYTES, 'HEX') AS BYTES_AS_HEX
FROM TEXT_BINARY_LAB
WHERE CUSTOMER_NAME ILIKE '%lee%';

UPDATE TEXT_BINARY_LAB
SET NOTES = NOTES || ' — verified'
WHERE ID = 1;

DELETE FROM TEXT_BINARY_LAB WHERE ID = 2;
SELECT * FROM TEXT_BINARY_LAB;
```

Source: [string and binary types](https://docs.snowflake.com/en/sql-reference/data-types-text).

## 6. Boolean and SQL NULL

`BOOLEAN` uses three-valued SQL logic: `TRUE`, `FALSE`, and unknown (`NULL`). A comparison with `NULL` is unknown, so use `IS NULL`, `IS NOT NULL`, or `IS [NOT] DISTINCT FROM`.

```sql
CREATE OR REPLACE TABLE BOOLEAN_TYPES_LAB (
  ID INTEGER,
  IS_ACTIVE BOOLEAN,
  SOURCE_VALUE VARCHAR
);

INSERT INTO BOOLEAN_TYPES_LAB VALUES
  (1, TRUE,  'yes'),
  (2, FALSE, 'off'),
  (3, NULL,  NULL);

SELECT * FROM BOOLEAN_TYPES_LAB WHERE IS_ACTIVE;
SELECT * FROM BOOLEAN_TYPES_LAB WHERE NOT IS_ACTIVE;
SELECT * FROM BOOLEAN_TYPES_LAB WHERE IS_ACTIVE IS NULL;

SELECT ID, SOURCE_VALUE, TRY_TO_BOOLEAN(SOURCE_VALUE) AS PARSED_BOOLEAN,
       IS_ACTIVE IS NOT DISTINCT FROM TRY_TO_BOOLEAN(SOURCE_VALUE) AS SAME_VALUE
FROM BOOLEAN_TYPES_LAB
ORDER BY ID;

UPDATE BOOLEAN_TYPES_LAB SET IS_ACTIVE = TRUE WHERE ID = 3;
DELETE FROM BOOLEAN_TYPES_LAB WHERE IS_ACTIVE = FALSE;
```

Explicit string conversion accepts values such as `true`, `false`, `yes`, `no`, `on`, `off`, `1`, and `0` without regard to case. Source: [logical types](https://docs.snowflake.com/en/sql-reference/data-types-logical).

## 7. Date, time, and timestamp types

- `DATE` stores a calendar date.
- `TIME` stores time of day without a date or time zone.
- `TIMESTAMP_NTZ` stores wall-clock date/time without time-zone semantics.
- `TIMESTAMP_LTZ` represents an instant and displays it in the session time zone. The original zone name is not stored.
- `TIMESTAMP_TZ` stores an instant with its numeric offset. It does not retain an IANA zone name and its future daylight-saving rules.

```sql
CREATE OR REPLACE TABLE DATETIME_TYPES_LAB (
  EVENT_ID INTEGER,
  BUSINESS_DATE DATE,
  LOCAL_TIME TIME(3),
  WALL_CLOCK TIMESTAMP_NTZ(9),
  EVENT_INSTANT TIMESTAMP_LTZ(9),
  OFFSET_TIMESTAMP TIMESTAMP_TZ(9)
);

INSERT INTO DATETIME_TYPES_LAB VALUES
  (1, '2026-09-07'::DATE, '13:45:30.123'::TIME,
   '2026-09-07 13:45:30.123456789'::TIMESTAMP_NTZ,
   '2026-09-07 08:15:30 +00:00'::TIMESTAMP_LTZ,
   '2026-09-07 13:45:30 +05:30'::TIMESTAMP_TZ),
  (2, '2026-09-08'::DATE, '09:00:00'::TIME,
   '2026-09-08 09:00:00'::TIMESTAMP_NTZ,
   '2026-09-08 03:30:00 +00:00'::TIMESTAMP_LTZ,
   '2026-09-07 20:30:00 -07:00'::TIMESTAMP_TZ);

SELECT EVENT_ID, BUSINESS_DATE, LOCAL_TIME, WALL_CLOCK,
       EVENT_INSTANT, OFFSET_TIMESTAMP,
       CONVERT_TIMEZONE('Asia/Kolkata', 'UTC', WALL_CLOCK) AS WALL_CLOCK_AS_UTC,
       DATE_PART('EPOCH_SECOND', EVENT_INSTANT) AS EPOCH_SECONDS
FROM DATETIME_TYPES_LAB
WHERE EVENT_INSTANT >= '2026-09-07 00:00:00 +00:00'::TIMESTAMP_LTZ
  AND EVENT_INSTANT <  '2026-09-09 00:00:00 +00:00'::TIMESTAMP_LTZ
ORDER BY EVENT_INSTANT;

UPDATE DATETIME_TYPES_LAB
SET EVENT_INSTANT = DATEADD('MINUTE', 30, EVENT_INSTANT)
WHERE EVENT_ID = 1;

DELETE FROM DATETIME_TYPES_LAB WHERE BUSINESS_DATE < CURRENT_DATE() - 3650;
```

Half-open ranges (`>= start AND < next_start`) avoid missing fractional seconds. `TIMESTAMP` is an alias controlled by `TIMESTAMP_TYPE_MAPPING`; use an explicit variant in durable DDL. Source: [date and time types](https://docs.snowflake.com/en/sql-reference/data-types-datetime).

## 8. One realistic scalar table

This table combines the types most business models use. Defaults are evaluated when a row is inserted. `UUID` receives a separate deep-dive later.

```sql
CREATE OR REPLACE TABLE TYPED_ORDER_HEADER (
  ORDER_ID NUMBER(12,0),
  EXTERNAL_ID UUID DEFAULT UUID_STRING(),
  CUSTOMER_NAME VARCHAR(100) NOT NULL,
  ORDER_TOTAL NUMBER(14,2) NOT NULL,
  DISCOUNT_RATE NUMBER(5,4),
  IS_PAID BOOLEAN DEFAULT FALSE,
  ORDER_DATE DATE,
  CREATED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  SOURCE_DIGEST BINARY(32)
);

INSERT INTO TYPED_ORDER_HEADER
  (ORDER_ID, CUSTOMER_NAME, ORDER_TOTAL, DISCOUNT_RATE, IS_PAID,
   ORDER_DATE, SOURCE_DIGEST)
VALUES
  (5001, 'Asha Rao', 1580.00, 0.0500, TRUE,  '2026-09-01', SHA2_BINARY('5001', 256)),
  (5002, 'Ben Lee',   450.00, NULL,   FALSE, '2026-09-02', SHA2_BINARY('5002', 256));

UPDATE TYPED_ORDER_HEADER
SET IS_PAID = TRUE, ORDER_TOTAL = ORDER_TOTAL - 25.00
WHERE ORDER_ID = 5002;

SELECT ORDER_ID, EXTERNAL_ID, CUSTOMER_NAME, ORDER_TOTAL,
       COALESCE(DISCOUNT_RATE, 0) AS DISCOUNT_RATE, IS_PAID,
       TO_VARCHAR(SOURCE_DIGEST, 'HEX') AS DIGEST_HEX
FROM TYPED_ORDER_HEADER
WHERE IS_PAID AND ORDER_DATE BETWEEN '2026-09-01' AND '2026-09-30'
ORDER BY ORDER_ID;

DELETE FROM TYPED_ORDER_HEADER
WHERE ORDER_ID = 5002 AND IS_PAID;
```

## 9. Semi-structured source data

The next document contains nested objects, arrays, mixed scalar types, a key with a space, and JSON `null`. Copy it into a file named `orders_nested.json` if you want a local sample, or use it directly in the following SQL.

```json
{
  "order_id": 7001,
  "status": "NEW",
  "channel": "web",
  "ordered_at": "2026-09-07T10:15:30Z",
  "customer": {
    "id": 101,
    "name": "Asha Rao",
    "loyalty_tier": "SILVER",
    "address": {"city": "Chennai", "country": "IN"}
  },
  "totals": {"subtotal": 1500.00, "tax": 270.00, "grand_total": 1770.00},
  "items": [
    {"sku": "LAPTOP", "qty": 1, "unit_price": 1200.00,
     "attributes": {"color": "silver", "warranty_months": 24},
     "discounts": ["FESTIVE50", "LOYALTY25"]},
    {"sku": "DOCK", "qty": 2, "unit_price": 150.00,
     "attributes": {"ports": 8}, "discounts": []}
  ],
  "tags": ["fragile", "express"],
  "delivery instructions": "Call before delivery",
  "coupon": null,
  "internal_note": "training row"
}
```

## 10. `VARIANT`, semi-structured `OBJECT`, and semi-structured `ARRAY`

A `VARIANT` holds a scalar, object, or array together with its internal type. An unparameterized `OBJECT` maps string keys to `VARIANT` values. An unparameterized `ARRAY` holds `VARIANT` elements. Their shape can differ by row.

```sql
CREATE OR REPLACE TABLE SEMISTRUCTURED_LAB (
  EVENT_ID INTEGER,
  PAYLOAD VARIANT,
  CUSTOMER_OBJECT OBJECT,
  TAG_ARRAY ARRAY,
  XML_DOCUMENT OBJECT,
  INGESTED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

INSERT INTO SEMISTRUCTURED_LAB
  (EVENT_ID, PAYLOAD, CUSTOMER_OBJECT, TAG_ARRAY)
SELECT 7001, P, P:customer::OBJECT, P:tags::ARRAY
FROM (
  SELECT PARSE_JSON($$
  {
    "order_id": 7001, "status": "NEW", "channel": "web",
    "ordered_at": "2026-09-07T10:15:30Z",
    "customer": {"id":101, "name":"Asha Rao", "loyalty_tier":"SILVER",
      "address":{"city":"Chennai", "country":"IN"}},
    "totals": {"subtotal":1500.00, "tax":270.00, "grand_total":1770.00},
    "items": [
      {"sku":"LAPTOP", "qty":1, "unit_price":1200.00,
       "attributes":{"color":"silver", "warranty_months":24},
       "discounts":["FESTIVE50","LOYALTY25"]},
      {"sku":"DOCK", "qty":2, "unit_price":150.00,
       "attributes":{"ports":8}, "discounts":[]}
    ],
    "tags":["fragile","express"],
    "delivery instructions":"Call before delivery",
    "coupon":null, "internal_note":"training row"
  }
  $$) AS P
);

INSERT INTO SEMISTRUCTURED_LAB
  (EVENT_ID, PAYLOAD, CUSTOMER_OBJECT, TAG_ARRAY)
SELECT 7002, P, P:customer::OBJECT, P:tags::ARRAY
FROM (
  SELECT PARSE_JSON($$
  {"order_id":7002, "status":"CANCELLED", "channel":"store",
   "customer":{"id":102, "name":"Ben Lee",
     "address":{"city":"Bengaluru", "country":"IN"}},
   "totals":{"grand_total":450.00},
   "items":[{"sku":"MONITOR","qty":1,"unit_price":450.00}],
   "tags":["store-pickup"]}
  $$) AS P
);

SELECT EVENT_ID, TYPEOF(PAYLOAD), TYPEOF(CUSTOMER_OBJECT), TYPEOF(TAG_ARRAY)
FROM SEMISTRUCTURED_LAB ORDER BY EVENT_ID;
```

Source: [semi-structured types](https://docs.snowflake.com/en/sql-reference/data-types-semistructured).

## 11. Navigate and filter nested structures

A colon begins path traversal from a column. Dots traverse object fields and brackets address array positions or unusual key names. Extracted semi-structured values remain `VARIANT`; cast them before numeric, temporal, or relational comparison. JSON key matching is case-sensitive.

```sql
SELECT
  EVENT_ID,
  PAYLOAD:order_id::INTEGER AS ORDER_ID,
  PAYLOAD:customer.name::VARCHAR AS CUSTOMER_NAME,
  PAYLOAD:customer.address.city::VARCHAR AS CITY,
  PAYLOAD:items[0].sku::VARCHAR AS FIRST_SKU,
  PAYLOAD['delivery instructions']::VARCHAR AS DELIVERY_INSTRUCTIONS,
  PAYLOAD:totals.grand_total::NUMBER(12,2) AS GRAND_TOTAL,
  TRY_TO_TIMESTAMP_TZ(PAYLOAD:ordered_at::VARCHAR) AS ORDERED_AT
FROM SEMISTRUCTURED_LAB
WHERE PAYLOAD:status::VARCHAR IN ('NEW', 'SHIPPED')
  AND PAYLOAD:totals.grand_total::NUMBER(12,2) > 1000
ORDER BY ORDER_ID;

SELECT EVENT_ID, GET_PATH(PAYLOAD, 'customer.address.country')::VARCHAR AS COUNTRY
FROM SEMISTRUCTURED_LAB
WHERE ARRAY_CONTAINS('fragile'::VARIANT, PAYLOAD:tags::ARRAY);
```

## 12. Expand nested arrays with `FLATTEN`

`LATERAL FLATTEN` correlates each source row with zero or more elements. Its useful columns include `INDEX`, `KEY`, `PATH`, `VALUE`, and `THIS`. Use a second lateral flatten for an array nested inside each item.

```sql
SELECT
  S.EVENT_ID,
  ITEM.INDEX AS ITEM_INDEX,
  ITEM.VALUE:sku::VARCHAR AS SKU,
  ITEM.VALUE:qty::INTEGER AS QUANTITY,
  ITEM.VALUE:unit_price::NUMBER(12,2) AS UNIT_PRICE,
  ITEM.VALUE:attributes.color::VARCHAR AS COLOR
FROM SEMISTRUCTURED_LAB AS S,
LATERAL FLATTEN(INPUT => S.PAYLOAD:items) AS ITEM
ORDER BY S.EVENT_ID, ITEM.INDEX;

SELECT
  S.EVENT_ID, ITEM.VALUE:sku::VARCHAR AS SKU,
  DISCOUNT.INDEX AS DISCOUNT_INDEX,
  DISCOUNT.VALUE::VARCHAR AS DISCOUNT_CODE
FROM SEMISTRUCTURED_LAB AS S,
LATERAL FLATTEN(INPUT => S.PAYLOAD:items) AS ITEM,
LATERAL FLATTEN(INPUT => ITEM.VALUE:discounts, OUTER => TRUE) AS DISCOUNT
WHERE S.EVENT_ID = 7001
ORDER BY ITEM.INDEX, DISCOUNT.INDEX;

SELECT S.EVENT_ID,
       SUM(ITEM.VALUE:qty::NUMBER * ITEM.VALUE:unit_price::NUMBER(12,2))
         AS CALCULATED_SUBTOTAL
FROM SEMISTRUCTURED_LAB AS S,
LATERAL FLATTEN(INPUT => S.PAYLOAD:items) AS ITEM
GROUP BY S.EVENT_ID
ORDER BY S.EVENT_ID;

SELECT DISTINCT S.EVENT_ID
FROM SEMISTRUCTURED_LAB AS S,
LATERAL FLATTEN(INPUT => S.PAYLOAD:items) AS ITEM
WHERE ITEM.VALUE:sku::VARCHAR = 'LAPTOP';

SELECT F.PATH, F.KEY, F.INDEX, TYPEOF(F.VALUE) AS VALUE_TYPE, F.VALUE
FROM SEMISTRUCTURED_LAB AS S,
LATERAL FLATTEN(INPUT => S.PAYLOAD, RECURSIVE => TRUE) AS F
WHERE S.EVENT_ID = 7001
ORDER BY F.PATH;
```

Source: [`FLATTEN`](https://docs.snowflake.com/en/sql-reference/functions/flatten).

## 13. Insert, update, and delete nested content

Semi-structured paths are read expressions, not assignable columns. Build a new object or array and assign it back to the column. The fourth `OBJECT_INSERT` argument permits replacement of an existing key.

```sql
-- Replace a top-level value.
UPDATE SEMISTRUCTURED_LAB
SET PAYLOAD = OBJECT_INSERT(PAYLOAD::OBJECT, 'status', 'SHIPPED', TRUE)
WHERE EVENT_ID = 7001;

-- Replace a value inside the nested customer object.
UPDATE SEMISTRUCTURED_LAB
SET PAYLOAD = OBJECT_INSERT(
  PAYLOAD::OBJECT,
  'customer',
  OBJECT_INSERT(PAYLOAD:customer::OBJECT, 'loyalty_tier', 'GOLD', TRUE),
  TRUE
)
WHERE EVENT_ID = 7001;

-- Replace the tags array with an appended array.
UPDATE SEMISTRUCTURED_LAB
SET PAYLOAD = OBJECT_INSERT(
  PAYLOAD::OBJECT, 'tags',
  ARRAY_APPEND(PAYLOAD:tags::ARRAY, 'priority'::VARIANT), TRUE
)
WHERE EVENT_ID = 7001;

-- Remove one key from the object.
UPDATE SEMISTRUCTURED_LAB
SET PAYLOAD = OBJECT_DELETE(PAYLOAD::OBJECT, 'internal_note')
WHERE EVENT_ID = 7001;

SELECT EVENT_ID, PAYLOAD:status, PAYLOAD:customer.loyalty_tier,
       PAYLOAD:tags, PAYLOAD:internal_note
FROM SEMISTRUCTURED_LAB ORDER BY EVENT_ID;

-- Delete a complete relational row using a nested predicate.
DELETE FROM SEMISTRUCTURED_LAB
WHERE PAYLOAD:status::VARCHAR = 'CANCELLED';
```

Sources: [`OBJECT_INSERT`](https://docs.snowflake.com/en/sql-reference/functions/object_insert), [`OBJECT_DELETE`](https://docs.snowflake.com/en/sql-reference/functions/object_delete), [`ARRAY_APPEND`](https://docs.snowflake.com/en/sql-reference/functions/array_append).

## 14. JSON null, missing fields, and schema drift

JSON `null` is a value inside `VARIANT`. SQL `NULL` means no SQL value. A missing path normally produces SQL `NULL`. `IS_NULL_VALUE` identifies a JSON null value.

```sql
SELECT
  EVENT_ID,
  PAYLOAD:coupon AS COUPON_VALUE,
  IS_NULL_VALUE(PAYLOAD:coupon) AS IS_JSON_NULL,
  PAYLOAD:missing_key IS NULL AS MISSING_IS_SQL_NULL,
  TYPEOF(PAYLOAD:coupon) AS COUPON_TYPE,
  OBJECT_KEYS(PAYLOAD) AS TOP_LEVEL_KEYS
FROM SEMISTRUCTURED_LAB;

SELECT
  CHECK_JSON('{"valid":1}') AS VALID_RETURNS_NULL,
  CHECK_JSON('{broken}') AS INVALID_RETURNS_MESSAGE,
  TRY_PARSE_JSON('{broken}') AS SAFE_PARSE_RETURNS_NULL;

SELECT
  IFF(IS_OBJECT(PAYLOAD), 'OBJECT', TYPEOF(PAYLOAD)) AS PAYLOAD_SHAPE,
  COALESCE(PAYLOAD:customer.loyalty_tier::VARCHAR, 'UNKNOWN') AS TIER
FROM SEMISTRUCTURED_LAB;

-- PARSE_JSON raises an error for invalid JSON. TRY_PARSE_JSON returns NULL.
-- SELECT PARSE_JSON('{broken}');
```

Good ingestion keeps the raw `VARIANT`, validates required paths, and promotes stable frequently-filtered fields into typed columns or downstream tables.

## 15. XML as semi-structured data

Copy this document to `order_nested.xml` for a file-based exercise, or paste the dollar-quoted version into the following SQL. XML attributes and element content use different access patterns.

```xml
<?xml version="1.0" encoding="UTF-8"?>
<order id="7001" status="SHIPPED">
  <customer>
    <name>Asha Rao</name>
    <city>Chennai</city>
  </customer>
  <items>
    <item sku="LAPTOP"><qty>1</qty><price>1200.00</price></item>
    <item sku="DOCK"><qty>2</qty><price>150.00</price></item>
  </items>
  <total currency="INR">1770.00</total>
</order>
```

```sql
UPDATE SEMISTRUCTURED_LAB
SET XML_DOCUMENT = PARSE_XML($$
<order id="7001" status="SHIPPED">
  <customer><name>Asha Rao</name><city>Chennai</city></customer>
  <items>
    <item sku="LAPTOP"><qty>1</qty><price>1200.00</price></item>
    <item sku="DOCK"><qty>2</qty><price>150.00</price></item>
  </items>
  <total currency="INR">1770.00</total>
</order>
$$, TRUE)
WHERE EVENT_ID = 7001;

SELECT TYPEOF(XML_DOCUMENT), TO_XML(XML_DOCUMENT) AS XML_TEXT
FROM SEMISTRUCTURED_LAB
WHERE EVENT_ID = 7001;
```

`PARSE_XML(..., TRUE)` disables automatic conversion so element text remains text until explicitly cast. Source: [`PARSE_XML`](https://docs.snowflake.com/en/sql-reference/functions/parse_xml).

## 16. Query and replace XML

`GET(xml, '@attribute')` reads an attribute. `XMLGET` returns a child element object; `GET(child, '$')` reads that element's content. The optional instance number selects repeated elements using zero-based indexing.

```sql
WITH X AS (
  SELECT EVENT_ID, XML_DOCUMENT,
         XMLGET(XML_DOCUMENT, 'customer') AS CUSTOMER_NODE,
         XMLGET(XML_DOCUMENT, 'items') AS ITEMS_NODE,
         XMLGET(XML_DOCUMENT, 'total') AS TOTAL_NODE
  FROM SEMISTRUCTURED_LAB
  WHERE EVENT_ID = 7001
)
SELECT
  GET(XML_DOCUMENT, '@id')::INTEGER AS ORDER_ID,
  GET(XML_DOCUMENT, '@status')::VARCHAR AS STATUS,
  GET(XMLGET(CUSTOMER_NODE, 'name'), '$')::VARCHAR AS CUSTOMER_NAME,
  GET(XMLGET(CUSTOMER_NODE, 'city'), '$')::VARCHAR AS CITY,
  GET(XMLGET(ITEMS_NODE, 'item', 0), '@sku')::VARCHAR AS FIRST_SKU,
  GET(XMLGET(XMLGET(ITEMS_NODE, 'item', 0), 'qty'), '$')::INTEGER AS FIRST_QTY,
  GET(TOTAL_NODE, '@currency')::VARCHAR AS CURRENCY,
  GET(TOTAL_NODE, '$')::NUMBER(12,2) AS TOTAL
FROM X;

-- XML path mutation is usually clearer as a document transformation upstream
-- or a replacement with a newly parsed document. This replaces the document.
UPDATE SEMISTRUCTURED_LAB
SET XML_DOCUMENT = PARSE_XML($$
<order id="7001" status="DELIVERED">
  <customer><name>Asha Rao</name><city>Chennai</city></customer>
  <total currency="INR">1770.00</total>
</order>
$$, TRUE)
WHERE EVENT_ID = 7001;
```

Source: [`XMLGET`](https://docs.snowflake.com/en/sql-reference/functions/xmlget).

## 17. Structured `OBJECT`, `ARRAY`, and `MAP`

Structured types preserve nested modeling while declaring the allowed shape. A structured object declares named fields, a structured array declares one element type, and a map declares key and value types. This gives earlier type checking and typed path results.

`INSERT ... SELECT` is used because `VALUES` does not support object/array constants and several constructors.

```sql
CREATE OR REPLACE TABLE STRUCTURED_TYPES_LAB (
  PERSON_ID INTEGER,
  ADDRESS OBJECT(
    street VARCHAR(100),
    city VARCHAR(50),
    postal_code VARCHAR(20),
    location OBJECT(lat FLOAT, lon FLOAT)
  ),
  SKILLS ARRAY(VARCHAR),
  PREFERENCES MAP(VARCHAR, VARCHAR)
);

INSERT INTO STRUCTURED_TYPES_LAB
SELECT
  1,
  {'street':'10 Lake Road', 'city':'Chennai', 'postal_code':'600001',
   'location':{'lat':13.0827, 'lon':80.2707}}
    ::OBJECT(street VARCHAR(100), city VARCHAR(50), postal_code VARCHAR(20),
             location OBJECT(lat FLOAT, lon FLOAT)),
  ['SQL', 'Python', 'Snowflake']::ARRAY(VARCHAR),
  {'language':'en', 'timezone':'Asia/Kolkata'}
    ::MAP(VARCHAR, VARCHAR);

INSERT INTO STRUCTURED_TYPES_LAB
SELECT
  2,
  {'street':'25 Market Street', 'city':'Bengaluru', 'postal_code':'560001',
   'location':{'lat':12.9716, 'lon':77.5946}}
    ::OBJECT(street VARCHAR(100), city VARCHAR(50), postal_code VARCHAR(20),
             location OBJECT(lat FLOAT, lon FLOAT)),
  ['SQL', 'dbt']::ARRAY(VARCHAR),
  {'language':'kn', 'timezone':'Asia/Kolkata'}
    ::MAP(VARCHAR, VARCHAR);

SELECT PERSON_ID, SYSTEM$TYPEOF(ADDRESS), SYSTEM$TYPEOF(SKILLS),
       SYSTEM$TYPEOF(PREFERENCES)
FROM STRUCTURED_TYPES_LAB;
```

Source: [structured types](https://docs.snowflake.com/en/sql-reference/data-types-structured).

## 18. Query and mutate structured values

A known structured field returns its declared type. Missing structured object fields are compile-time errors; a missing map key returns `NULL`. `OBJECT_INSERT`, `ARRAY_APPEND`, and `MAP_INSERT` return new values for assignment.

```sql
SELECT PERSON_ID,
       ADDRESS:city AS CITY,
       ADDRESS:location.lat AS LATITUDE,
       SKILLS[0] AS PRIMARY_SKILL,
       GET(PREFERENCES, 'language') AS LANGUAGE,
       MAP_CONTAINS_KEY('timezone', PREFERENCES) AS HAS_TIMEZONE
FROM STRUCTURED_TYPES_LAB
WHERE ARRAY_CONTAINS('Snowflake'::VARCHAR, SKILLS)
ORDER BY PERSON_ID;

UPDATE STRUCTURED_TYPES_LAB
SET ADDRESS = OBJECT_INSERT(ADDRESS, 'city', 'Pune', TRUE),
    SKILLS = ARRAY_APPEND(SKILLS, 'Data Modeling'),
    PREFERENCES = MAP_INSERT(PREFERENCES, 'theme', 'dark', TRUE)
WHERE PERSON_ID = 1;

SELECT PERSON_ID, ADDRESS, SKILLS, PREFERENCES
FROM STRUCTURED_TYPES_LAB ORDER BY PERSON_ID;

DELETE FROM STRUCTURED_TYPES_LAB WHERE PERSON_ID = 2;

-- Invalid: ADDRESS has no declared field named province.
-- SELECT ADDRESS:province FROM STRUCTURED_TYPES_LAB;
```

## 19. `GEOGRAPHY` and `GEOMETRY`

`GEOGRAPHY` models longitude/latitude on Earth and returns distances in meters. `GEOMETRY` uses a planar coordinate system and units defined by its spatial reference. WKT writes a point as `POINT(longitude latitude)`.

```sql
CREATE OR REPLACE TABLE GEOSPATIAL_TYPES_LAB (
  PLACE_ID INTEGER,
  PLACE_NAME VARCHAR,
  EARTH_LOCATION GEOGRAPHY,
  PLANAR_LOCATION GEOMETRY
);

INSERT INTO GEOSPATIAL_TYPES_LAB VALUES
  (1, 'Chennai',   TO_GEOGRAPHY('POINT(80.2707 13.0827)'),
                   TO_GEOMETRY('POINT(80.2707 13.0827)')),
  (2, 'Bengaluru', TO_GEOGRAPHY('POINT(77.5946 12.9716)'),
                   TO_GEOMETRY('POINT(77.5946 12.9716)')),
  (3, 'Hyderabad', TO_GEOGRAPHY('POINT(78.4867 17.3850)'),
                   TO_GEOMETRY('POINT(78.4867 17.3850)'));

SELECT PLACE_NAME, ST_ASWKT(EARTH_LOCATION) AS WKT,
       ROUND(ST_DISTANCE(
         EARTH_LOCATION, TO_GEOGRAPHY('POINT(77.5946 12.9716)')
       ) / 1000, 1) AS KM_FROM_BENGALURU
FROM GEOSPATIAL_TYPES_LAB
WHERE ST_DWITHIN(
  EARTH_LOCATION, TO_GEOGRAPHY('POINT(77.5946 12.9716)'), 650000
)
ORDER BY KM_FROM_BENGALURU;

SELECT ST_AREA(TO_GEOMETRY(
  'POLYGON((0 0, 10 0, 10 5, 0 5, 0 0))'
)) AS PLANAR_SQUARE_UNITS;

UPDATE GEOSPATIAL_TYPES_LAB
SET EARTH_LOCATION = TO_GEOGRAPHY('POINT(80.2500 13.0500)')
WHERE PLACE_ID = 1;

DELETE FROM GEOSPATIAL_TYPES_LAB WHERE PLACE_ID = 3;
```

Do not compare longitude/latitude using ordinary Euclidean arithmetic. Keep objects in compatible types and spatial reference systems. Source: [geospatial types](https://docs.snowflake.com/en/sql-reference/data-types-geospatial).

## 20. `UUID`

`UUID` stores a 128-bit identifier. Snowflake accepts the standard 8-4-4-4-12 text representation, but the column is a UUID type. The type does not enforce uniqueness; add a data-quality rule or key appropriate to the table family.

```sql
CREATE OR REPLACE TABLE UUID_TYPES_LAB (
  ENTITY_ID UUID DEFAULT UUID_STRING() NOT NULL,
  ENTITY_NAME VARCHAR(100)
);

INSERT INTO UUID_TYPES_LAB (ENTITY_NAME) VALUES ('generated-id');
INSERT INTO UUID_TYPES_LAB VALUES
  ('c73d9175-0a1d-48c6-8d30-df165461328b', 'literal-id');

SELECT ENTITY_ID, ENTITY_NAME
FROM UUID_TYPES_LAB
WHERE ENTITY_ID = TO_UUID('C73D9175-0A1D-48C6-8D30-DF165461328B');

UPDATE UUID_TYPES_LAB
SET ENTITY_ID = UUID_STRING()
WHERE ENTITY_NAME = 'literal-id';

DELETE FROM UUID_TYPES_LAB WHERE ENTITY_NAME = 'generated-id';
SELECT * FROM UUID_TYPES_LAB;
```

Some drivers treat UUID values as strings, and UUID is not supported in every table/runtime family. Source: [UUID type](https://docs.snowflake.com/en/sql-reference/data-types-uuid).

## 21. `VECTOR`

A vector has a fixed dimension and either 32-bit integer or 32-bit floating-point elements. Cast an array to create it. Use vector similarity functions for semantic comparison; raw vector ordering is byte-wise and has no distance meaning.

```sql
CREATE OR REPLACE TABLE VECTOR_TYPES_LAB (
  ITEM_ID INTEGER,
  LABEL VARCHAR,
  EMBEDDING VECTOR(FLOAT, 3)
);

INSERT INTO VECTOR_TYPES_LAB
SELECT 1, 'red',   [1.0, 0.0, 0.0]::VECTOR(FLOAT, 3)
UNION ALL
SELECT 2, 'pink',  [0.9, 0.1, 0.0]::VECTOR(FLOAT, 3)
UNION ALL
SELECT 3, 'blue',  [0.0, 0.0, 1.0]::VECTOR(FLOAT, 3);

WITH QUERY_VECTOR AS (
  SELECT [1.0, 0.0, 0.0]::VECTOR(FLOAT, 3) AS V
)
SELECT ITEM_ID, LABEL,
       VECTOR_COSINE_SIMILARITY(EMBEDDING, V) AS COSINE_SIMILARITY,
       VECTOR_L2_DISTANCE(EMBEDDING, V) AS L2_DISTANCE
FROM VECTOR_TYPES_LAB, QUERY_VECTOR
WHERE VECTOR_COSINE_SIMILARITY(EMBEDDING, V) >= 0.75
ORDER BY COSINE_SIMILARITY DESC;

UPDATE VECTOR_TYPES_LAB
SET EMBEDDING = [0.8, 0.2, 0.0]::VECTOR(FLOAT, 3)
WHERE ITEM_ID = 2;

DELETE FROM VECTOR_TYPES_LAB WHERE ITEM_ID = 3;

-- Invalid dimension: the column requires exactly three elements.
-- INSERT INTO VECTOR_TYPES_LAB SELECT 4, 'bad', [1.0, 2.0]::VECTOR(FLOAT, 3);
```

Direct file loading of a vector column is not supported; load an array and cast it. Source: [vector types](https://docs.snowflake.com/en/sql-reference/data-types-vector).

## 22. `FILE` — an unstructured-file reference

A `FILE` value stores metadata and a reference to an internal or external staged file. It does not copy the file bytes into the table. Deleting the row does not delete the staged object, and removing the staged object does not update the stored reference.

The table creation below is self-contained. The insert requires the directory-enabled `@STAGE_LAB_DB.PUBLIC.LAB_FILES` stage created in `D381_StagePractical.ipynb`.

```sql
CREATE OR REPLACE TABLE FILE_TYPES_LAB (
  RELATIVE_PATH VARCHAR,
  DOCUMENT FILE
);

-- Run these statements after completing D381.
-- ALTER STAGE STAGE_LAB_DB.PUBLIC.LAB_FILES REFRESH;
-- INSERT INTO FILE_TYPES_LAB
-- SELECT RELATIVE_PATH, TO_FILE(FILE_URL)
-- FROM DIRECTORY(@STAGE_LAB_DB.PUBLIC.LAB_FILES)
-- WHERE RELATIVE_PATH LIKE 'json/%' OR RELATIVE_PATH LIKE 'xml/%';

-- SELECT RELATIVE_PATH,
--        FL_GET_CONTENT_TYPE(DOCUMENT) AS CONTENT_TYPE,
--        FL_GET_SIZE(DOCUMENT) AS SIZE_BYTES,
--        FL_GET_STAGE(DOCUMENT) AS STAGE_NAME,
--        FL_GET_RELATIVE_PATH(DOCUMENT) AS FILE_PATH
-- FROM FILE_TYPES_LAB;

-- Replacing a FILE column changes only the reference.
-- DELETE FROM FILE_TYPES_LAB WHERE RELATIVE_PATH LIKE 'json/%';
```

A `FILE` column cannot be used in `GROUP BY` or `ORDER BY`, among other limitations. Source: [unstructured and FILE types](https://docs.snowflake.com/en/sql-reference/data-types-unstructured).

## 23. User-defined types

A user-defined type is a schema object based on one existing Snowflake type. It gives a domain a reusable name and definition. It is not a constraint: a type based on `VARCHAR(320)` does not validate email syntax, for example.

```sql
CREATE TYPE IF NOT EXISTS POSTAL_ADDRESS AS OBJECT(
  street VARCHAR(100),
  city VARCHAR(50),
  country_code CHAR(2),
  postal_code VARCHAR(20)
) COMMENT = 'Reusable postal address domain';

CREATE OR REPLACE TABLE UDT_CUSTOMER_LAB (
  CUSTOMER_ID INTEGER,
  CUSTOMER_NAME VARCHAR(100),
  ADDRESS POSTAL_ADDRESS
);

INSERT INTO UDT_CUSTOMER_LAB
SELECT 1, 'Asha Rao',
  {'street':'10 Lake Road', 'city':'Chennai',
   'country_code':'IN', 'postal_code':'600001'}::POSTAL_ADDRESS;

SELECT CUSTOMER_ID, CUSTOMER_NAME, ADDRESS:city AS CITY,
       ADDRESS:postal_code AS POSTAL_CODE, SYSTEM$TYPEOF(ADDRESS) AS ADDRESS_TYPE
FROM UDT_CUSTOMER_LAB
WHERE ADDRESS:country_code = 'IN';

UPDATE UDT_CUSTOMER_LAB
SET ADDRESS = OBJECT_INSERT(ADDRESS, 'city', 'Pune', TRUE)::POSTAL_ADDRESS
WHERE CUSTOMER_ID = 1;

SHOW TYPES LIKE 'POSTAL_ADDRESS' IN SCHEMA D39_TABLE_LAB.DATATYPES;
DESCRIBE TYPE POSTAL_ADDRESS;
```

Change a type carefully: replacing or dropping it can invalidate statements that access dependent columns. Source: [user-defined types](https://docs.snowflake.com/en/sql-reference/data-types-user-defined).

## 24. `UNKNOWN`, aliases, and migration boundaries

`UNKNOWN` is not a general-purpose column type. It is an Iceberg v3 placeholder used when a more specific type is not known. It occupies no storage and reads as SQL `NULL`. Study it with the Iceberg material in `D398_IcebergTableLifecycle.ipynb`; do not declare it in a standard Snowflake table.

Common migration rules:

| Source-system idea | Snowflake design |
|---|---|
| Different integer widths | Use `NUMBER(p,0)` when a range contract matters; integer aliases themselves resolve to `NUMBER(38,0)` |
| JSON or JSONB column | Use `VARIANT`, then promote stable business fields to typed columns |
| Arbitrary byte array/BLOB | Use `BINARY`; use a stage plus `FILE` for large unstructured documents |
| Zoned timestamp with named region | Store the instant plus a separate IANA zone-name column when future regional rules matter |
| Enum/domain validation | Use a column constraint or governed reference data; a user-defined type alone does not constrain values |
| Duration/interval column | Store an agreed unit as `NUMBER`, or start/end timestamps; interval expressions are not a portable stored business type |

Avoid choosing a type from its alias name alone. Run `DESCRIBE TABLE` and inspect how Snowflake resolved it.

## 25. Bridge to D381 staged JSON, XML, Avro, and Parquet

The D381 sample formats all represent orders, but parser output and destination types differ. JSON, Avro, and Parquet records arrive through `$1` as semi-structured values. XML arrives as Snowflake's XML object representation. CSV fields arrive as strings and should be converted into typed destination columns.

```sql
-- Run after D381 has created and populated its stage and file formats.
CREATE OR REPLACE TABLE D381_JSON_TYPED (
  RAW_RECORD VARIANT,
  ORDER_ID INTEGER,
  CUSTOMER VARCHAR(100),
  CITY VARCHAR(100),
  AMOUNT NUMBER(10,2),
  ITEMS ARRAY
);

-- INSERT INTO D381_JSON_TYPED
-- SELECT t.$1,
--        t.$1:order_id::INTEGER,
--        t.$1:customer::VARCHAR,
--        t.$1:city::VARCHAR,
--        t.$1:amount::NUMBER(10,2),
--        t.$1:items::ARRAY
-- FROM @STAGE_LAB_DB.PUBLIC.LAB_FILES/json/
--   (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_JSON_ORDERS',
--    PATTERN => '.*orders[.]json$') t;

-- SELECT J.ORDER_ID, I.VALUE:sku::VARCHAR AS SKU, I.VALUE:qty::INTEGER AS QTY
-- FROM D381_JSON_TYPED J, LATERAL FLATTEN(INPUT => J.ITEMS) I;
```

For the supplied D381 files, use these destination choices:

| File | Raw reader | Recommended typed projection |
|---|---|---|
| CSV | `$1`, `$2`, ... strings | `INTEGER`, `VARCHAR`, `NUMBER(10,2)` |
| JSON | `$1` semi-structured record | Preserve raw `VARIANT`; project paths with casts |
| XML | `$1` XML object | Preserve object; use `GET`/`XMLGET` and explicit casts |
| Avro | `$1` decoded record | Cast named fields; retain raw `VARIANT` when auditability matters |
| Parquet | `$1` decoded record | Respect embedded schema, then cast to the business contract |

The inline files in this notebook make the core data-type lab independent. D381 remains the practical for `PUT`, stage paths, file formats, and `COPY INTO`.

## 26. Type-selection rules

1. Model business money with a deliberate `NUMBER(p,s)`, not `FLOAT`.
2. Use explicit timestamp variants. Store an instant in `TIMESTAMP_LTZ` and retain a zone name separately when required.
3. Keep raw semi-structured input for replay, then expose stable fields as typed columns.
4. Use a structured nested type when the shape is a contract; use semi-structured types when shape flexibility is intentional.
5. Use `GEOGRAPHY` for Earth locations and `GEOMETRY` for planar coordinate systems.
6. Use `BINARY` for bytes stored in a row and `FILE` for a staged-file reference.
7. Use `UUID` for distributed identifiers and add an explicit uniqueness mechanism when required.
8. Compare vectors with similarity or distance functions.
9. Profile incoming strings with `TRY_TO_*`, then reject or quarantine invalid rows rather than silently losing them.
10. Inspect resolved definitions with `DESCRIBE TABLE` and expressions with `SYSTEM$TYPEOF`.

## 27. Practice tasks

1. Insert a numeric row whose unit price has three decimal places. Predict and verify the stored scale.
2. Change the session time zone to `America/Los_Angeles` and compare the LTZ, NTZ, and TZ displays.
3. Add a third JSON order with no `coupon` key. Write a query that distinguishes missing, JSON null, and a real coupon string.
4. Add two more line items, then calculate order totals with `FLATTEN`.
5. Add a nested `shipments` array and flatten shipments and their tracking events.
6. Try to insert a numeric value into the structured `SKILLS` array. Explain the error.
7. Find all places within 400 km of Bengaluru with `ST_DWITHIN`.
8. Rank all vectors against `[0.5, 0.5, 0.0]`. Compare cosine similarity with L2 distance.
9. After completing D381, load its JSON into `D381_JSON_TYPED` and verify three orders and three item rows.
10. Design a typed production table from the raw order JSON. Justify every promoted column and every field left in `VARIANT`.

## 28. Optional cleanup

Drop tables before the user-defined type that a table references. This cleanup leaves the shared D39 database, schema, and warehouse available for later notebooks.

```sql
USE ROLE SYSADMIN;
USE DATABASE D39_TABLE_LAB;
USE SCHEMA DATATYPES;

DROP TABLE IF EXISTS D381_JSON_TYPED;
DROP TABLE IF EXISTS FILE_TYPES_LAB;
DROP TABLE IF EXISTS UDT_CUSTOMER_LAB;
DROP TABLE IF EXISTS VECTOR_TYPES_LAB;
DROP TABLE IF EXISTS UUID_TYPES_LAB;
DROP TABLE IF EXISTS GEOSPATIAL_TYPES_LAB;
DROP TABLE IF EXISTS STRUCTURED_TYPES_LAB;
DROP TABLE IF EXISTS SEMISTRUCTURED_LAB;
DROP TABLE IF EXISTS TYPED_ORDER_HEADER;
DROP TABLE IF EXISTS DATETIME_TYPES_LAB;
DROP TABLE IF EXISTS BOOLEAN_TYPES_LAB;
DROP TABLE IF EXISTS TEXT_BINARY_LAB;
DROP TABLE IF EXISTS NUMERIC_TYPES_LAB;

BEGIN
  DROP TYPE POSTAL_ADDRESS;
EXCEPTION
  WHEN OTHER THEN NULL;
END;

-- Only after completing every D39 notebook:
-- DROP DATABASE D39_TABLE_LAB;
-- DROP WAREHOUSE D39_LAB_WH;
```

## 29. Official references

- [Data-type summary](https://docs.snowflake.com/en/sql-reference/intro-summary-data-types)
- [Numeric](https://docs.snowflake.com/en/sql-reference/data-types-numeric), [string and binary](https://docs.snowflake.com/en/sql-reference/data-types-text), [logical](https://docs.snowflake.com/en/sql-reference/data-types-logical), and [date/time](https://docs.snowflake.com/en/sql-reference/data-types-datetime)
- [Semi-structured](https://docs.snowflake.com/en/sql-reference/data-types-semistructured) and [structured](https://docs.snowflake.com/en/sql-reference/data-types-structured) types
- [`FLATTEN`](https://docs.snowflake.com/en/sql-reference/functions/flatten), [`PARSE_XML`](https://docs.snowflake.com/en/sql-reference/functions/parse_xml), and [`XMLGET`](https://docs.snowflake.com/en/sql-reference/functions/xmlget)
- [Geospatial](https://docs.snowflake.com/en/sql-reference/data-types-geospatial), [FILE](https://docs.snowflake.com/en/sql-reference/data-types-unstructured), [UUID](https://docs.snowflake.com/en/sql-reference/data-types-uuid), and [VECTOR](https://docs.snowflake.com/en/sql-reference/data-types-vector)
- [User-defined types](https://docs.snowflake.com/en/sql-reference/data-types-user-defined)

Syntax and feature coverage were checked against Snowflake documentation on 2026-09-07. Newer types can depend on account rollout, table family, client, and runtime support. The SQL has been statically reviewed but requires a Snowflake account for live execution.